# Evaluating LLMs: Metrics, Benchmarks, Judges and Error Bars

The runnable code of [Evaluating LLMs: Metrics, Benchmarks, Judges and Error Bars](https://residual-stream.lvtan71.workers.dev/posts/evaluating-llms/), in the order the post runs it. Each
code cell is followed by the output the post printed, so you can compare.

[![Open in Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/lvtan71/residual-stream-code/blob/main/notebooks/evaluating-llms.ipynb)

**Hardware:** Use a GPU runtime (Colab: Runtime → Change runtime type → T4 GPU). About 5 min end to end.


The post's numbers come from an Apple M4 laptop. Other hardware and library
versions can change the last digits of timings and of some floating-point
results; the conclusions should not change.

In [ ]:
# Setup: fetch the helpers and data this post's code uses into a working folder.
import os, shutil, subprocess, sys
SLUG = "evaluating-llms"
ROOT = os.path.abspath("..") if os.path.isdir("../shared") else os.path.abspath("residual-stream-code")
if not os.path.isdir(ROOT):
    subprocess.run(["git", "clone", "-q", "--depth", "1", "https://github.com/lvtan71/residual-stream-code", ROOT], check=True)
WORK = os.path.join(ROOT, "work", SLUG)
os.makedirs(WORK, exist_ok=True)
for f in []:
    shutil.copy(os.path.join(ROOT, f), WORK)
    if f.endswith(".tar.gz"):                       # the blog's posts, as of that run
        shutil.unpack_archive(os.path.join(WORK, os.path.basename(f)), WORK)
os.chdir(WORK)
sys.path.insert(0, WORK)
print("working in", WORK)

## How many questions?

[Read this part in the post](https://residual-stream.lvtan71.workers.dev/posts/evaluating-llms/#how-many-questions)

In [ ]:
import math
from statistics import NormalDist

# How many questions does an eval need? Two models, accuracy around 70%,
# and a difference of 2 points to detect, with a 5% false-alarm rate
# (two-sided) and 80% power.
z = NormalDist().inv_cdf
za, zb = z(0.975), z(0.80)
p1, p2 = 0.70, 0.72
# Unpaired: the two models answer different random questions.
n_unpaired = (za + zb) ** 2 * (p1 * (1 - p1) + p2 * (1 - p2)) / (p2 - p1) ** 2
print(f"unpaired (different questions): {math.ceil(n_unpaired):,} questions per model")
# Paired: both answer the same questions, and only the questions they
# disagree on carry information (McNemar). If they disagree on a share d of
# questions, the per-question difference has variance d - (p2 - p1)^2,
# about d. (Independent errors would make them disagree on
# p1 (1 - p2) + p2 (1 - p1) = 41%: the unpaired case.)
for d in (0.30, 0.15, 0.08):
    n = (za + zb) ** 2 * d / (p2 - p1) ** 2
    print(f"paired, models disagree on {d:.0%} of questions: {math.ceil(n):,} questions")

# A 95% confidence interval for one accuracy (Wilson score interval).
def wilson(k, n, conf=0.95):
    q = z(0.5 + conf / 2)
    p = k / n
    centre = (p + q * q / (2 * n)) / (1 + q * q / n)
    half = q * math.sqrt(p * (1 - p) / n + q * q / (4 * n * n)) / (1 + q * q / n)
    return centre - half, centre + half
print()
for k, n in ((70, 100), (700, 1000), (7000, 10000)):
    lo, hi = wilson(k, n)
    print(f"{k:,} right of {n:,}: accuracy {k / n:.0%}, 95% interval {lo:.1%} to {hi:.1%} "
          f"(about ±{(hi - lo) / 2 * 100:.1f} points)")

# What matters for a comparison is the interval on the difference.
n = 500
unpaired = za * math.sqrt((p1 * (1 - p1) + p2 * (1 - p2)) / n)
paired = za * math.sqrt((0.15 - (p2 - p1) ** 2) / n)
print(f"\n{n} questions, 95% interval on the difference in accuracy: unpaired ±{unpaired * 100:.1f} points, "
      f"paired (disagreeing on 15%) ±{paired * 100:.1f} points")

# Cohen's kappa: agreement between two annotators beyond what chance gives.
# 100 answers labeled good/bad by two people:
#                  B good   B bad
#   A good           45       10
#   A bad             5       40
a = [[45, 10], [5, 40]]
n = sum(map(sum, a))
p_obs = (a[0][0] + a[1][1]) / n
pa_good, pb_good = sum(a[0]) / n, (a[0][0] + a[1][0]) / n
p_chance = pa_good * pb_good + (1 - pa_good) * (1 - pb_good)
kappa = (p_obs - p_chance) / (1 - p_chance)
print(f"\nCohen's kappa: observed agreement {p_obs:.2f}, chance agreement {pa_good:.2f} x {pb_good:.2f} + "
      f"{1 - pa_good:.2f} x {1 - pb_good:.2f} = {p_chance:.3f}, kappa = ({p_obs:.2f} - {p_chance:.3f}) / (1 - {p_chance:.3f}) = {kappa:.2f}")

The post printed:

```text
unpaired (different questions): 8,077 questions per model
paired, models disagree on 30% of questions: 5,887 questions
paired, models disagree on 15% of questions: 2,944 questions
paired, models disagree on 8% of questions: 1,570 questions

70 right of 100: accuracy 70%, 95% interval 60.4% to 78.1% (about ±8.8 points)
700 right of 1,000: accuracy 70%, 95% interval 67.1% to 72.8% (about ±2.8 points)
7,000 right of 10,000: accuracy 70%, 95% interval 69.1% to 70.9% (about ±0.9 points)

500 questions, 95% interval on the difference in accuracy: unpaired ±5.6 points, paired (disagreeing on 15%) ±3.4 points

Cohen's kappa: observed agreement 0.85, chance agreement 0.55 x 0.50 + 0.45 x 0.50 = 0.500, kappa = (0.85 - 0.500) / (1 - 0.500) = 0.70
```

## LLM-as-judge, measured

[Read this part in the post](https://residual-stream.lvtan71.workers.dev/posts/evaluating-llms/#llm-as-judge-measured)

In [ ]:
import random
import torch
from transformers import AutoModelForCausalLM, AutoTokenizer
from huggingface_hub import snapshot_download
GPU = "cuda" if torch.cuda.is_available() else "mps" if torch.backends.mps.is_available() else "cpu"  # notebook: the post ran on "mps"

# Qwen2.5-1.5B-Instruct as a pairwise judge, on questions whose right answer
# is known, so its biases can be measured rather than argued about. The
# judge sees a question and two answers and is asked which is correct; its
# choice is whichever of the tokens "A" and "B" it rates more likely next.
PATH = snapshot_download("Qwen/Qwen2.5-1.5B-Instruct",
                         revision="989aa7980e4cf806f80c7fef2b1adb7bc71aa306")
tok = AutoTokenizer.from_pretrained(PATH)
model = AutoModelForCausalLM.from_pretrained(PATH, dtype=torch.float16).to(GPU).eval()
A, B = tok.encode("A")[0], tok.encode("B")[0]

@torch.no_grad()
def judge(question, first, second, labels=("A", "B")):
    """Returns 1 if the judge prefers the first-shown answer, else 2.
    labels: the letters printed before the first and second answers."""
    prompt = (f"Question: {question}\n\nAnswer {labels[0]}: {first}\n\nAnswer {labels[1]}: {second}\n\n"
              "Which answer is correct? Reply with a single letter, A or B.")
    ids = tok.apply_chat_template([{"role": "user", "content": prompt}], add_generation_prompt=True,
                                  return_tensors="pt", return_dict=True)["input_ids"].to(GPU)
    logits = model(ids).logits[0, -1]
    first_letter = A if labels[0] == "A" else B
    second_letter = B if labels[0] == "A" else A
    return 1 if logits[first_letter] > logits[second_letter] else 2

rng = random.Random(0)
items = []                                   # (question, right answer, wrong answer)
for _ in range(100):
    a, b = rng.randint(12, 99), rng.randint(12, 99)
    wrong = a * b + rng.choice([-1, 1]) * rng.randint(1, 9) * 10
    items.append((f"What is {a} x {b}?", str(a * b), str(wrong)))
capitals = [("Australia", "Canberra", "Sydney"), ("Canada", "Ottawa", "Toronto"), ("Brazil", "Brasilia", "Rio de Janeiro"),
            ("Turkey", "Ankara", "Istanbul"), ("Switzerland", "Bern", "Zurich"), ("Morocco", "Rabat", "Casablanca"),
            ("Nigeria", "Abuja", "Lagos"), ("Vietnam", "Hanoi", "Ho Chi Minh City"), ("New Zealand", "Wellington", "Auckland"),
            ("South Africa", "Pretoria", "Johannesburg"), ("Pakistan", "Islamabad", "Karachi"), ("Myanmar", "Naypyidaw", "Yangon"),
            ("Kazakhstan", "Astana", "Almaty"), ("Tanzania", "Dodoma", "Dar es Salaam"), ("Bolivia", "Sucre", "Santa Cruz"),
            ("Ecuador", "Quito", "Guayaquil"), ("India", "New Delhi", "Mumbai"), ("China", "Beijing", "Shanghai"),
            ("Italy", "Rome", "Milan"), ("Germany", "Berlin", "Munich")]
items += [(f"What is the capital of {c}?", r, w) for c, r, w in capitals]

def verbose(wrong):            # a wrong answer with claims of having been checked
    return (f"The answer is {wrong}. I worked this out carefully, checking each step twice, and the result is "
            f"consistent with standard references, so you can rely on {wrong} being correct.")
def padded(ans):               # the same length of padding, with no claims
    return (f"The answer is {ans}. This question asks for one value, and the reply gives that value here in "
            f"a full sentence rather than on its own, followed by this sentence, which adds nothing to {ans}.")

def rate(pairs, labels=("A", "B")):
    """Share of judgments that pick the right answer, over both orders.
    pairs: (question, answer that is right, the other answer)."""
    hits = [judge(q, r, w, labels) == 1 for q, r, w in pairs] + [judge(q, w, r, labels) == 2 for q, r, w in pairs]
    return sum(hits) / len(hits)

subsets = {"multiplication": items[:100], "capitals": items[100:]}
print(f"{len(items)} questions with a known answer: 100 two-digit multiplications, {len(capitals)} capitals\n")
print(f"{'':40s} {'multiplication':>15s} {'capitals':>9s}")
def row(name, f):
    print(f"{name:40s} " + " ".join(f"{f(v):>{w}.0%}" for v, w in zip(subsets.values(), (15, 9))), flush=True)
row("right answer first: judge right", lambda p: sum(judge(q, r, w) == 1 for q, r, w in p) / len(p))
row("right answer second: judge right", lambda p: sum(judge(q, w, r) == 2 for q, r, w in p) / len(p))
row("  same, first answer labeled B", lambda p: sum(judge(q, w, r, ("B", "A")) == 2 for q, r, w in p) / len(p))
row("right in both orders", lambda p: sum(judge(q, r, w) == 1 and judge(q, w, r) == 2 for q, r, w in p) / len(p))
row("chose the same slot both times", lambda p: sum(judge(q, r, w) == judge(q, w, r) for q, r, w in p) / len(p))
print("\nshare of judgments picking the right answer, both orders averaged:")
row("bare right vs bare wrong", lambda p: rate(p))
row("bare right vs wrong + claims of checking", lambda p: rate([(q, r, verbose(w)) for q, r, w in p]))
row("bare right vs wrong + neutral padding", lambda p: rate([(q, r, padded(w)) for q, r, w in p]))
row("right + neutral padding vs bare wrong", lambda p: rate([(q, padded(r), w) for q, r, w in p]))

The post printed:

```text
120 questions with a known answer: 100 two-digit multiplications, 20 capitals

                                          multiplication  capitals
right answer first: judge right                      99%       85%
right answer second: judge right                     12%      100%
  same, first answer labeled B                        0%       55%
right in both orders                                 12%       85%
chose the same slot both times                       87%       15%

share of judgments picking the right answer, both orders averaged:
bare right vs bare wrong                             56%       92%
bare right vs wrong + claims of checking              6%       42%
bare right vs wrong + neutral padding                48%       82%
right + neutral padding vs bare wrong                62%       95%
```

## Arenas: ratings from comparisons

[Read this part in the post](https://residual-stream.lvtan71.workers.dev/posts/evaluating-llms/#arenas-ratings-from-comparisons)

In [ ]:
import numpy as np

# Arena-style ratings: people compare two anonymous models' answers and vote
# for one. The Bradley-Terry model ("RLHF") says model i beats model j with
# probability sigmoid(r_i - r_j); fit the ratings r by maximum likelihood
# and report them on the Elo-like scale arenas use (400 / ln 10 per unit).
# Four models with known true ratings; votes on random pairs; 95% intervals
# from 200 bootstrap resamples of the votes.
rng = np.random.default_rng(0)
true = np.array([0.0, 0.10, 0.20, 0.60])               # in logits: small, small, small, large gaps
names = ["model A", "model B", "model C", "model D"]
SCALE = 400 / np.log(10)

def fit(i, j, win, iters=500, lr=0.5):
    """Maximum likelihood by gradient ascent; model A pinned at 0."""
    r = np.zeros(len(true))
    for _ in range(iters):
        p = 1 / (1 + np.exp(-(r[i] - r[j])))
        g = np.zeros_like(r)
        np.add.at(g, i, win - p); np.add.at(g, j, p - win)
        r += lr * g / len(win) * len(true)
        r -= r[0]
    return r

for votes in (500, 5000, 50000):
    i = rng.integers(0, 4, votes); j = (i + rng.integers(1, 4, votes)) % 4
    win = (rng.random(votes) < 1 / (1 + np.exp(-(true[i] - true[j])))).astype(float)
    r = fit(i, j, win)
    boot = []
    for _ in range(200):
        k = rng.integers(0, votes, votes)
        boot.append(fit(i[k], j[k], win[k]))
    lo, hi = np.percentile(boot, [2.5, 97.5], axis=0)
    print(f"{votes:,} votes:")
    for n, t, est, a, b in zip(names, true, r, lo, hi):
        print(f"  {n}: true {t * SCALE:5.0f}, estimated {est * SCALE:5.0f}, 95% interval {a * SCALE:5.0f} to {b * SCALE:5.0f}")

The post printed:

```text
500 votes:
  model A: true     0, estimated     0, 95% interval     0 to     0
  model B: true    17, estimated    44, 95% interval    -9 to    93
  model C: true    35, estimated    51, 95% interval    -2 to   104
  model D: true   104, estimated   133, 95% interval    73 to   184
5,000 votes:
  model A: true     0, estimated     0, 95% interval     0 to     0
  model B: true    17, estimated    19, 95% interval     5 to    38
  model C: true    35, estimated    33, 95% interval    16 to    53
  model D: true   104, estimated   102, 95% interval    87 to   120
50,000 votes:
  model A: true     0, estimated     0, 95% interval     0 to     0
  model B: true    17, estimated    17, 95% interval    11 to    22
  model C: true    35, estimated    30, 95% interval    25 to    36
  model D: true   104, estimated   100, 95% interval    95 to   105
```